# CIFAR-10 data extraction

This example follows the same layout as LeakPro's MIA and model-inversion examples: `train_config.yaml` controls target preparation, `audit.yaml` defines the attacks, `cifar10_handler.py` connects user-owned objects to LeakPro, and this notebook runs the experiment.

The audit covers Carlini et al. Section 5.1 for unconditional CIFAR-10 models and SIDE Algorithm 1 for small diffusion models. Run it only against a model and reference data you are authorized to audit. The default `smoke` profile checks the full workflow; it does not reproduce the papers' extraction rates.


In [ ]:
import json
import os
import sys
import time
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torchvision
import yaml

candidates = [Path.cwd(), *Path.cwd().parents]
example_dir = next(
    (path / 'examples' / 'extraction' / 'cifar10' for path in candidates if (path / 'examples' / 'extraction' / 'cifar10' / 'audit.yaml').exists()),
    Path.cwd(),
)
if not (example_dir / 'train_config.yaml').exists():
    raise FileNotFoundError('Run this notebook from the LeakPro checkout or examples/extraction/cifar10 directory.')
repo_root = example_dir.parents[2]
sys.path.insert(0, str(repo_root))
sys.path.insert(0, str(example_dir))
os.chdir(example_dir)

from leakpro import LeakPro
from cifar10_handler import CIFAR10ExtractionHandler, load_audit_config
from cifar10_model import (
    RunProfile,
    load_cifar10,
    make_adapter,
    make_feature_extractor,
    seed_everything,
    select_device,
    sha256_file,
    sha256_mapping,
    sha256_module_state,
    sha256_tensor,
    train_or_load_target,
)

train_config = yaml.safe_load(Path('train_config.yaml').read_text(encoding='utf-8'))
profile_name = os.getenv('LEAKPRO_CIFAR_PROFILE', train_config['run']['profile'])
if profile_name not in train_config['profiles']:
    raise ValueError(f'Unknown profile {profile_name!r}; choose from {sorted(train_config["profiles"])}.')
profile_values = dict(train_config['profiles'][profile_name])
profile = RunProfile(name=profile_name, seed=int(train_config['run']['random_seed']), **profile_values)
audit_config_value = os.getenv(
    'LEAKPRO_CIFAR_AUDIT_CONFIG',
    train_config['run']['audit_configs'][profile_name],
)
audit_config_path = Path(audit_config_value)
device = select_device(os.getenv('LEAKPRO_CIFAR_DEVICE', train_config['run']['device']))
seed_everything(profile.seed)
data_dir = Path(os.getenv('LEAKPRO_CIFAR_DATA_DIR', train_config['run']['data_dir']))
target_dir = Path(train_config['run']['target_dir']) / profile.name
target_dir.mkdir(parents=True, exist_ok=True)
print({
    'profile': profile.name,
    'device': str(device),
    'data_dir': str(data_dir),
    'target_dir': str(target_dir),
    'audit_config': str(audit_config_path),
})


## Prepare the target data

The target trains on the deterministic prefix selected by `train_config.yaml`. The same images form the authorized reference set for near-copy verification. CIFAR-10 class labels are not passed to the unconditional target or Carlini attack; SIDE builds its own labels by clustering generated images.


In [ ]:
train_dataset, reference_images = load_cifar10(profile, target_dir, data_dir)
assert reference_images.shape == (profile.reference_size, 3, 32, 32)
assert reference_images.dtype == torch.float32
assert torch.isfinite(reference_images).all()
assert float(reference_images.min()) >= -1.0 and float(reference_images.max()) <= 1.0

preview = reference_images[:8].add(1.0).div(2.0)
figure, axes = plt.subplots(1, len(preview), figsize=(12, 2))
for axis, image in zip(axes, preview):
    axis.imshow(image.permute(1, 2, 0))
    axis.axis('off')
figure.suptitle(f'CIFAR-10 target subset, profile={profile.name}')
plt.show()


## Train the diffusion target

The target minimizes the noise-prediction loss

$$\mathcal{L}=\mathbb{E}_{x_0,t,\epsilon}\left[\lVert\epsilon-\epsilon_\theta(x_t,t)\rVert_2^2\right],$$

where $x_t=\sqrt{\bar\alpha_t}x_0+\sqrt{1-\bar\alpha_t}\epsilon$. Sampling uses deterministic DDIM with the reverse-step count set by the selected profile. A checkpoint is reused only when its stored profile and format version match.


In [ ]:
checkpoint_path = target_dir / f'cifar10_ddpm_{profile.name}.pt'
force_retrain_env = os.getenv('LEAKPRO_CIFAR_FORCE_RETRAIN')
force_retrain = force_retrain_env == '1' if force_retrain_env is not None else bool(train_config['run']['force_retrain'])
if force_retrain and checkpoint_path.exists():
    checkpoint_path.unlink()

training_started = time.perf_counter()
model, diffusion, checkpoint_path, epoch_losses = train_or_load_target(
    profile, train_dataset, target_dir, device
)
training_seconds = time.perf_counter() - training_started
checkpoint_hash = sha256_file(checkpoint_path)
if epoch_losses:
    plt.figure(figsize=(6, 3))
    plt.plot(epoch_losses)
    plt.xlabel('Epoch')
    plt.ylabel('Noise-prediction MSE')
    plt.title('Target training loss')
    plt.show()

adapter = make_adapter(model, diffusion, device)
samples = adapter.sample(4, conditions=None, seed=profile.seed + 10)
repeated = adapter.sample(4, conditions=None, seed=profile.seed + 10)
assert samples.shape == (4, 3, 32, 32)
assert torch.isfinite(samples).all()
torch.testing.assert_close(samples, repeated)

figure, axes = plt.subplots(1, 4, figsize=(7, 2))
for axis, image in zip(axes, samples.detach().cpu().add(1.0).div(2.0)):
    axis.imshow(image.permute(1, 2, 0).clamp(0.0, 1.0))
    axis.axis('off')
figure.suptitle('Unguided DDIM samples')
plt.show()
print({'checkpoint': str(checkpoint_path), 'sha256': checkpoint_hash, 'training_seconds': round(training_seconds, 2)})


## Run the LeakPro audit

`CIFAR10ExtractionHandler` plays the same role as the user input handlers in the MIA and model-inversion examples. It gives LeakPro the trained target adapter, authorized references, and SIDE feature model. The selected checked-in audit YAML owns every attack parameter. The notebook changes only the runtime target fingerprint, then writes that resolved config beside the target checkpoint.


In [ ]:
feature_extractor, feature_transform = make_feature_extractor()
identity_components = {
    'target_checkpoint_sha256': checkpoint_hash,
    'model_source_sha256': sha256_file(Path('cifar10_model.py')),
    'handler_source_sha256': sha256_file(Path('cifar10_handler.py')),
    'side_feature_state_sha256': sha256_module_state(feature_extractor),
    'side_feature_transform': 'resize-224-bilinear-align-corners-false-imagenet-normalization-v1',
    'authorized_references_sha256': sha256_tensor(reference_images),
}
target_fingerprint = f"sha256:{sha256_mapping(identity_components)}"
CIFAR10ExtractionHandler.configure(
    adapter=adapter,
    references=reference_images,
    feature_extractor=feature_extractor,
    feature_transform=feature_transform,
)

audit_config = load_audit_config(
    audit_config_path,
    target_fingerprint=target_fingerprint,
)
attack_configs = {entry['attack']: entry for entry in audit_config['audit']['attack_list']}
carlini_config = attack_configs['carlini_diffusion']
side_config = attack_configs['side']
audit_output = Path(audit_config['audit']['output_dir'])
runtime_config_path = target_dir / 'audit.yaml'
runtime_config_path.write_text(yaml.safe_dump(audit_config, sort_keys=False), encoding='utf-8')

audit_started = time.perf_counter()
results = LeakPro(CIFAR10ExtractionHandler, str(runtime_config_path)).run_audit()
audit_seconds = time.perf_counter() - audit_started
carlini_result, side_result = results
print({'audit_seconds': round(audit_seconds, 2), 'result_ids': [result.id for result in results]})


## Inspect the results

Zero Carlini candidates is a valid smoke result. It means no generated image passed the adaptive reference-neighborhood score. SIDE returns its guided generations and records the nearest authorized reference for each image. Stored SIDE L2 distances use the configured `[-1, 1]` coordinates.


In [ ]:
assert len(results) == 2
assert carlini_result.metrics['mode'] == 'unconditional_reference_audit'
assert carlini_result.metrics['images_generated'] == carlini_config['num_unconditional_generations']
assert side_result.metrics['images_generated'] == side_config['num_generations']
assert side_result.metrics['retained_clusters'] >= 2
assert all(torch.isfinite(torch.tensor(side_result.metrics['classifier_epoch_losses'])))
assert side_result.execution_trace[-1]['guidance_calls'] > 0
for result in results:
    result_dir = audit_output / 'results' / result.id
    assert (result_dir / 'result.json').exists()
    assert (result_dir / 'candidates.npz').exists()

def show_nearest_matches(result, title, maximum=6):
    records = [record for record in result.candidates if record.nearest_reference_index is not None]
    records.sort(key=lambda record: record.nearest_reference_distance)
    records = records[:maximum]
    if not records:
        print(f'{title}: no qualifying candidates')
        return
    figure, axes = plt.subplots(len(records), 2, figsize=(4, 2 * len(records)), squeeze=False)
    for row, record in enumerate(records):
        generated = result.images[record.image_index].detach().cpu()
        reference = reference_images[record.nearest_reference_index].add(1.0).div(2.0)
        axes[row, 0].imshow(generated.permute(1, 2, 0).clamp(0.0, 1.0))
        axes[row, 0].set_title(f'generated, d={record.nearest_reference_distance:.4f}')
        axes[row, 1].imshow(reference.permute(1, 2, 0).clamp(0.0, 1.0))
        axes[row, 1].set_title(f'train #{record.nearest_reference_index}')
        axes[row, 0].axis('off')
        axes[row, 1].axis('off')
    figure.suptitle(title)
    figure.tight_layout()
    plt.show()

print('Carlini metrics:', carlini_result.metrics)
print('SIDE metrics:', side_result.metrics)
show_nearest_matches(carlini_result, 'Carlini qualifying candidates')
show_nearest_matches(side_result, 'SIDE guided samples and nearest training references')


## Save the run manifest

The manifest records the resolved configs and identities needed to distinguish runs. It contains no raw training images.

The `smoke` profile verifies data loading, target training, checkpointing, DDIM sampling, Carlini scoring, SIDE clustering and classifier guidance, persistence, and visualization. It is not extraction-performance evidence. The `demonstration` profile trains longer on a smaller subset, but still uses less compute than either paper. Carlini et al. generated about one million unconditional CIFAR-10 candidates. SIDE reports roughly 2,048 target-training epochs, 10,000 evaluation generations, 100 clusters, cohesion threshold `0.5`, and SSCD features. This example substitutes ImageNet ResNet-18 features and records that substitution in the target identity.


In [ ]:
manifest = {
    'profile': asdict(profile),
    'checkpoint': str(checkpoint_path),
    'checkpoint_sha256': checkpoint_hash,
    'target_fingerprint': target_fingerprint,
    'identity_components': identity_components,
    'source_audit_config': str(audit_config_path),
    'runtime_audit_config': str(runtime_config_path),
    'device': str(device),
    'torch': torch.__version__,
    'torchvision': torchvision.__version__,
    'training_seconds': training_seconds,
    'audit_seconds': audit_seconds,
    'audit_config': audit_config,
    'result_ids': [result.id for result in results],
}
manifest_path = target_dir / 'run_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding='utf-8')
print(manifest_path)
